---

## Introducción al proyecto

Este proyecto tiene como objetivo analizar la rentabilidad de distintos planes de telecomunicaciones a partir del comportamiento real de los clientes. A través del análisis de datos de consumo (como llamadas, uso de internet y mensajes) se busca identificar qué planes generan mayores ingresos y bajo qué condiciones.

El análisis se centra en comparar el desempeño de los planes en función del uso de los usuarios, evaluando cómo las diferencias en consumo impactan directamente en la rentabilidad. Esto permite entender no solo qué plan es más rentable en general, sino también para qué tipo de cliente resulta más conveniente.

Los resultados obtenidos pueden utilizarse para optimizar la oferta comercial, ajustar estrategias de precios y tomar decisiones basadas en datos para maximizar ingresos y mejorar la segmentación de clientes.

## Inicialización

In [ ]:
import pandas as pd
import numpy as np
from scipy import stats as st
from matplotlib import pyplot as plt
from math import factorial

## Cargar datos

In [ ]:
df_calls = pd.read_csv('/datasets/megaline_calls.csv')
df_internet = pd.read_csv('/datasets/megaline_internet.csv')
df_messages = pd.read_csv('/datasets/megaline_messages.csv')
df_plans = pd.read_csv('/datasets/megaline_plans.csv')
df_users = pd.read_csv('/datasets/megaline_users.csv')

## Preparar los datos

## Tarifas

In [ ]:
# Imprimimos la información general sobre las tarifas
df_plans.info()

In [ ]:
# Imprimimos los datos de las tarifas

print(df_plans)

Tras revisar la estructura del dataset de planes, se observa que los tipos de datos son consistentes con la naturaleza de cada variable, por lo que no es necesario realizar conversiones adicionales.

No se identifican valores ausentes ni duplicados en este conjunto de datos. Sin embargo, se pueden aplicar algunos ajustes para mejorar la consistencia y legibilidad del dataframe. Por ejemplo, estandarizar los nombres de las columnas para mantener un formato uniforme, así como simplificar nombres como "mb_per_month_included" a "mb_included" para facilitar su interpretación.

Adicionalmente, se podría considerar utilizar el nombre del plan como índice para una organización más clara de la información.

## Corregir datos

In [ ]:
# Estandarizamos los nombres de las columnas
new_cols_names = []
for name in df_plans.columns:
    new_name = name.strip().lower().replace(' ', '_')
    new_cols_names.append(new_name)
df_plans.columns = new_cols_names

df_plans = df_plans.set_index('plan_name') # Indexar por tarifa
df_plans.rename(columns={'mb_per_month_included':'mb_included'}, inplace=True)# Cambiar nombre
print(df_plans) # Verificamos

## Enriquecer los datos

Se propone la creación de una nueva variable que capture el costo asociado al consumo excedente de datos (USD por MB adicional). Esta variable puede ser relevante en etapas posteriores del análisis, ya que permite evaluar con mayor precisión el impacto del uso adicional en la rentabilidad de cada plan.

Asimismo, se incorpora una columna que representa los GB incluidos en cada plan, con el objetivo de facilitar comparaciones y análisis posteriores relacionados con el consumo de datos.

In [ ]:
df_plans['usd_per_mb'] = df_plans['usd_per_gb']/1024
df_plans['gb_included'] = df_plans['mb_included']/1024
print(df_plans) # Verificamos

## Usuarios/as

In [ ]:
# Imprimimos la información general sobre los usuarios
df_users.info()

In [ ]:
# Imprimimos una muestra de datos para los usuarios
print(df_users.sample(n=10))

Se identificaron inconsistencias en los tipos de datos de las columnas de fechas, por lo que se procede a convertirlas a formato datetime para asegurar un manejo adecuado en el análisis.

En cuanto a valores ausentes, únicamente la columna *churn_date* presenta registros nulos, lo cual es consistente con clientes que permanecen activos al momento de la extracción de los datos.

Adicionalmente, se detectaron inconsistencias en el formato de la columna *city*, donde se mezclan mayúsculas y minúsculas. Se estandariza el formato para garantizar uniformidad.

Finalmente, se normalizan los nombres de las columnas con el objetivo de mantener consistencia y facilitar el procesamiento posterior de los datos.

### Corregir los datos

In [ ]:
# Estandarizamos los nombres de las columnas
new_cols_names = []
for name in df_users.columns:
    new_name = name.strip().lower().replace(' ', '_')
    new_cols_names.append(new_name)
df_users.columns = new_cols_names

# Convertimos columnas a tipo datetime
df_users['reg_date'] = pd.to_datetime(df_users['reg_date'])
df_users['churn_date'] = pd.to_datetime(df_users['churn_date'])

# Estandarizamos la columna city
df_users['city'] = df_users['city'].str.lower()

# Verificamos valores duplicados
print('valores duplicados:', df_users.duplicated().sum())

# Comprobamos
print()
df_users.info()
print()
print(df_users.sample(n=10))

### Enriquecer los datos

Se evaluó el tratamiento de los valores ausentes en la columna *churn_date*. En lugar de imputarlos con una etiqueta textual, se opta por crear una variable derivada que indique si el cliente continúa activo o no, en función de la presencia o ausencia de esta fecha.

Adicionalmente, se extrae información temporal a partir de *churn_date*, como el mes de cancelación, lo que permite enriquecer el análisis y facilitar la identificación de patrones en el comportamiento de los usuarios.

In [ ]:
df_users['active'] = df_users['churn_date'].isna() #mostrar si el usuario sigue activo
df_users['churn_month'] = df_users['churn_date'].dt.month #mostar el mes que dejaron de usar el servicio en número
df_users['churn_month_name'] = df_users['churn_date'].dt.month_name() #mostar el mes que dejaron de usar el servicio en texto
df_users.info()
print()
print(df_users.sample(n=10))

## Llamadas

In [ ]:
# Imprimmos la información general sobre las llamadas
df_calls.info()

In [ ]:
# Imprimimos una muestra de datos para las llamadas
print(df_calls.sample(n=10))

Se realizan ajustes en los tipos de datos para asegurar su correcta interpretación: la columna *call_date* se convierte a formato datetime y la columna *id* se transforma a tipo entero, eliminando caracteres innecesarios para conservar únicamente su valor numérico.

Asimismo, se renombra la columna *id* a *call_id* con el objetivo de mejorar la claridad y trazabilidad de los registros.

La variable de duración se mantiene en formato float, ya que permite representar fracciones de minuto. No obstante, considerando que el sistema de facturación redondea los segundos a minutos completos, este aspecto se toma en cuenta para el análisis posterior.

En cuanto a la calidad de los datos, no se identifican valores ausentes, mientras que la revisión de duplicados se abordará en una etapa posterior. Finalmente, se estandarizan los nombres de las columnas para mantener consistencia en el dataset.

### Corregir los datos

In [ ]:
# Estandarizamos los nombres de las columnas
new_cols_names = []
for name in df_calls.columns:
    new_name = name.strip().lower().replace(' ', '_')
    new_cols_names.append(new_name)
df_calls.columns = new_cols_names

# Cambiamos tipo de datos de la columna call_date
df_calls['call_date'] = pd.to_datetime(df_calls['call_date'])
# Corregimos y cambiamos tipo de datos de la columna id
df_calls['id'] = df_calls['id'].str.split('_').str[1].astype(int)
# Renombramos columna id
df_calls.rename(columns={'id':'call_id'}, inplace=True)
# Redondeamos columna duration a minutos
df_calls['duration'] = df_calls['duration'].apply(np.ceil).astype(int)
# Revisamos duplicados
print('Valores duplicados:', df_calls.duplicated().sum())

# Verificamos
print()
df_calls.info()
print()
print(df_calls.sample(n=10))

### Enriquecer los datos

Se extrae el mes a partir de la columna *call_date* con el objetivo de analizar la distribución y el volumen de llamadas a lo largo del tiempo.

In [ ]:
df_calls['call_month'] = df_calls['call_date'].dt.month #muestro el mes en número (para mayor facilidad en los análisis)
df_calls['call_month_name'] = df_calls['call_date'].dt.month_name() #muestro el mes en texto (para mayor flexibilidad)
df_calls.info()
print()
print(df_calls.sample(n=10)) #Verificamos

## Mensajes

In [ ]:
# Imprimimos la información general sobre los mensajes
df_messages.info()

In [ ]:
# Imprimimos una muestra de datos para los mensajes
print(df_messages.sample(n=10))

Se corrige el formato de la columna *id*, eliminando caracteres innecesarios para conservar únicamente su valor numérico y posteriormente convertirla a tipo entero. Además, se renombra a *sms_id* para mejorar la claridad.

La columna *message_date* se convierte a formato datetime para permitir su correcto análisis temporal.

En cuanto a la calidad de los datos, no se identifican valores ausentes. La revisión de duplicados se realiza en una etapa posterior. Finalmente, se estandarizan los nombres de las columnas para mantener consistencia en el dataset.

### Corregir los datos

In [ ]:
# Estandarizamos los nombres de las columnas
new_cols_names = []
for name in df_messages.columns:
    new_name = name.strip().lower().replace(' ', '_')
    new_cols_names.append(new_name)
df_messages.columns = new_cols_names

df_messages['id'] = df_messages['id'].str.split('_').str[1].astype(int) # Corregimos y convertimos tipos de datos de la columna id
df_messages.rename(columns={'id':'sms_id'}, inplace=True) # Cambiamos nombre de la columna id
df_messages['message_date'] = pd.to_datetime(df_messages['message_date']) # Convertimos columna message_date a datetime
print('Valores duplicados', df_messages.duplicated().sum())    # Revisamos valores duplicados
# Verificamos
print()
df_messages.info()
print()
print(df_messages.sample(n=10))

### Enriquecer los datos

Se extrae el mes a partir de la columna correspondiente para su uso en análisis temporal, manteniendo consistencia con el tratamiento aplicado en etapas anteriores.

In [ ]:
df_messages['message_month'] = df_messages['message_date'].dt.month # Mostramos el mes con número
df_messages['message_month_name'] = df_messages['message_date'].dt.month_name() # Mostramos el mes con texto

df_messages.info()
print()
print(df_messages.sample(n=10))

## Internet

In [ ]:
# Imprimimos la información general sobre el internet
df_internet.info()

In [ ]:
# Imprimimos una muestra de datos para el tráfico de internet
print(df_internet.sample(n=10))

Se corrige el formato de la columna *id*, eliminando caracteres innecesarios y convirtiéndola a tipo entero. Asimismo, se renombra para mejorar la claridad.

La columna *session_date* se transforma a formato datetime para su correcto análisis temporal.

Finalmente, se contempla la revisión de duplicados y valores ausentes como parte del proceso de validación de la calidad de los datos.

### Corregir los datos

In [ ]:
# Estandarizamos los nombres de las columnas
new_cols_names = []
for name in df_internet.columns:
    new_name = name.strip().lower().replace(' ', '_')
    new_cols_names.append(new_name)
df_internet.columns = new_cols_names

# Corregimos columna id y cambiar su tipo a int
df_internet['id'] = df_internet['id'].str.split('_').str[1].astype(int)
# Cambiamos nombre de la columna id
df_internet.rename(columns={'id':'session_id'}, inplace=True)
# Cambiamos columna session_date a datetime
df_internet['session_date'] = pd.to_datetime(df_internet['session_date'])
# Revisamos duplicados
print('Valores duplicados', df_internet.duplicated().sum())
# Verificamos
print()
df_internet.info()
print()
print(df_internet.sample(n=10))

### Enriquecer los datos

Se extrae el mes a partir de la columna *session_date* para su uso en análisis temporal del comportamiento de las sesiones.

Adicionalmente, se crea una variable que representa el volumen de datos utilizados por sesión, lo que permite enriquecer el análisis del consumo de los usuarios.

In [ ]:
df_internet['session_month'] = df_internet['session_date'].dt.month    # Mostramos el mes con número
df_internet['session_month_name'] = df_internet['session_date'].dt.month_name()    # Mostramos el mes con texto
df_internet['gb_used'] = df_internet['mb_used']/1024  # Mostramos gb usados en la sesión

df_internet.info()
print()
print(df_internet.sample(n=10))

Se estandarizan los valores de las columnas en todos los dataframes, eliminando espacios en blanco al inicio y al final de cada registro, con el fin de asegurar consistencia y evitar errores en el procesamiento y análisis de los datos.

In [ ]:
# Guardamos los dataframes con sus nombres con pares clave valor en un diccionario
dataframes = {'df_users':df_users, 'df_calls':df_calls, 'df_messages':df_messages, 'df_internet':df_internet}

for name, df in dataframes.items(): # Recorremos ese diccionario
    for col in df.columns:
        if df[col].dtype == 'object':
            df[col] = df[col].str.strip() # Estandarizamos los valores dentro de todas las columnas de tipo object de todos los dataframes

**Revisión de valores ausentes exlícitos**

In [ ]:
print('Valores ausentes tipo NaN en df_users:\n', df_users.isna().sum(), '\n')
print('Valores ausentes tipo NaN en df_calls:\n', df_calls.isna().sum(), '\n')
print('Valores ausentes tipo NaN en df_messages:\n', df_messages.isna().sum(), '\n')
print('Valores ausentes tipo NaN en df_internet:\n', df_internet.isna().sum(), '\n')

**Revisión de valores ausentes implícitos**

Se realiza una evaluación más detallada de posibles valores ausentes implícitos, con el objetivo de identificar inconsistencias que no se reflejan directamente como valores nulos, pero que pueden afectar el análisis.

In [ ]:
# Revisamos valores únicos en df_users
print('Valores únicos en df_users:')
for col in df_users.columns:
    print(f'Valores únicos en {col}:\n', df_users[col].value_counts(dropna=False, ascending=True), '\n')

# Revisamos valores únicos en df_calls
print('Valores únicos en df_calls:')
for col in df_calls.columns:
    print(f'Valores únicos en {col}:\n', df_calls[col].value_counts(dropna=False, ascending=True), '\n')

# Revisamos valores únicos en df_messages
print('Valores únicos en df_messages:')
for col in df_messages.columns:
    print(f'Valores únicos en {col}:\n', df_messages[col].value_counts(dropna=False, ascending=True), '\n')

# Revisamos valores únicos en df_internet
print('Valores únicos en df_internet:')
for col in df_internet.columns:
    print(f'Valores únicos en {col}:\n', df_internet[col].value_counts(dropna=False, ascending=True), '\n')

# Agregamos el ascending=True para que los valores que menos se repiten, los cuales son los más raros, aparezcan hasta arriba.

Se identifican valores igual a cero en las variables *duration* (df_calls) y *mb_used* (df_internet), lo cual sugiere la ausencia de información relevante sobre la duración de llamadas y el consumo de datos en ciertas observaciones.

Dado que estos registros no aportan información útil para el análisis del comportamiento de uso, se decide eliminarlos para evitar distorsiones en los resultados y garantizar una mayor calidad en el dataset.

In [ ]:
# Eliminamos filas con el número cero en df_calls['duration'] y en df_internet['mb_used']
df_calls = df_calls[df_calls['duration'] > 0]
df_internet = df_internet[df_internet['mb_used'] > 0]
# Verificamos:
print(df_calls[df_calls['duration'] == 0])
print(df_internet[df_internet['mb_used'] == 0])

**Revisión de filas completamente duplicadas**:

In [ ]:
# Revisamos filas completamente duplicadas en cada uno de los dataframes a excepción de df_plans
for name, df in dataframes.items(): # Recorremos el diccionario anterior
    print(f'Filas duplicadas en {name}:\n', df.duplicated().sum(), '\n')

## Estudiar las condiciones de las tarifas

In [ ]:
# Imprimimos las condiciones de la tarifa
print(df_plans)

In [ ]:
# Calculamos el número de llamadas hechas por cada usuario al mes y guardamos el resultado.
df_monthly_calls = df_calls.groupby(['user_id', 'call_month'])['call_id'].count().reset_index()
print(df_monthly_calls.head(20))

In [ ]:
# Calculamos la cantidad de minutos usados por cada usuario al mes y guardamos el resultado.

df_monthly_minutes = df_calls.groupby(['user_id', 'call_month'])['duration'].sum().reset_index()
print(df_monthly_minutes)

In [ ]:
# Calculamos el número de mensajes enviados por cada usuario al mes y guardamos el resultado.
df_monthly_messages = df_messages.groupby(['user_id', 'message_month'])['sms_id'].count().reset_index()
print(df_monthly_messages)

In [ ]:
# Calculamos el volumen del tráfico de Internet usado por cada usuario al mes y guardamos el resultado.
# Redondeamos el total de gb usados al mes ya que esto es lo que se va a considerar para cobrar
df_monthly_gb = df_internet.groupby(['user_id', 'session_month'])['gb_used'].sum().apply(np.ceil).astype(int).reset_index()
print(df_monthly_gb)

In [ ]:
# Fusionamos los datos de llamadas, minutos, mensajes e Internet con base en user_id y month

# Estandarizamos los nombres de las columnas de mes en los distintos dataframes para poder hacer la fusión
df_monthly_calls = df_monthly_calls.rename(columns={'call_month': 'month'})
df_monthly_minutes = df_monthly_minutes.rename(columns={'call_month': 'month'})
df_monthly_messages = df_monthly_messages.rename(columns={'message_month': 'month'})
df_monthly_gb = df_monthly_gb.rename(columns={'session_month': 'month'})

# Realizamos la fusión de los cautro dataframes por su usuario y mes
df_merged = df_monthly_calls.merge(df_monthly_minutes, on=['user_id', 'month'])
df_merged = df_merged.merge(df_monthly_messages, on=['user_id', 'month'])
users_consumption = df_merged.merge(df_monthly_gb, on=['user_id', 'month'])
print(users_consumption.head(10))
print()
users_consumption.info()

In [ ]:
# Añadimos la información de la tarifa
# Restablecemos el índice del dataframe de planes para facilitar su integración con el resto de los datos.
df_plans_reset = df_plans.reset_index()

# Estandarizamos el nombre de la columna de plan para permitir la unión entre dataframes
df_users = df_users.rename(columns={'plan':'plan_name'})

# Fusionamos la información de usuarios con los planes tarifarios
users_and_plans = df_users.merge(df_plans_reset, on='plan_name')

# Integramos la información de consumo con los datos de usuarios y planes
consumption_and_plans = users_consumption.merge(users_and_plans, on='user_id')

# Visualizamos y verificamos los resultados del dataframe final
print(consumption_and_plans.head(10))
print()
consumption_and_plans.info()

In [ ]:
# Calculamos el ingreso mensual para cada usuario

#POR USUARIO

#1. Minutos excedidos = (número total de minutos gastados) - (límite del paquete gratuito)
#Costo por exceso de minutos = (Minutos excedidos) * (costo por minuto excedido)

#2. Mensajes excedidos = (número total de mensajes enviados) - (límite del paquete gratuito)
#Costo por exceso de mensajes = (Mensajes excedidos) * (costo por mensaje excedido)

#3. GB excedidos = (número total de GB usados) - (límite del paquete gratuito)
#Costo por exceso de GB = (GB excedidos) * (costo por GB excedido)

#4.Cuota Mensual

#5.Ingreso total mensual = (Costo por exceso de minutos) + (Costo por exceso de mensajes) + (Costo por exceso de GB) + (Cuota Mensual)

#Minutos
consumption_and_plans['exceeded_minutes'] = np.where(consumption_and_plans['duration'] > consumption_and_plans['minutes_included'],
                                                    consumption_and_plans['duration'] - consumption_and_plans['minutes_included'],
                                                    0)
consumption_and_plans['extra_charge_per_minutes'] = (consumption_and_plans['exceeded_minutes']) * (consumption_and_plans['usd_per_minute'])
#Mensajes
consumption_and_plans['exceeded_messages'] = np.where(consumption_and_plans['sms_id'] > consumption_and_plans['messages_included'],
                                                    consumption_and_plans['sms_id'] - consumption_and_plans['messages_included'],
                                                    0)
consumption_and_plans['extra_charge_per_messages'] = (consumption_and_plans['exceeded_messages']) * (consumption_and_plans['usd_per_message'])
#GB
consumption_and_plans['exceeded_gb'] = np.where(consumption_and_plans['gb_used'] > consumption_and_plans['gb_included'],
                                                    consumption_and_plans['gb_used'] - consumption_and_plans['gb_included'],
                                                    0)
consumption_and_plans['extra_charge_per_gb'] = (consumption_and_plans['exceeded_gb']) * (consumption_and_plans['usd_per_gb'])

#monthly_fee = usd_monthly_pay

consumption_and_plans['monthly_income'] = consumption_and_plans['usd_monthly_pay'] + consumption_and_plans['extra_charge_per_minutes'] + consumption_and_plans['extra_charge_per_messages'] + consumption_and_plans['extra_charge_per_gb']

consumption_and_plans.info()
print()
print(consumption_and_plans.head(10))

## Estudiar el comportamiento del usuario

### Llamadas

In [ ]:
# Comparamos la duración promedio de llamadas por cada plan y por cada mes. Trazamos un gráfico de barras para visualizarla.

#1. Filtramos por plan
#2. Agrupamos por mes
surf_plan = consumption_and_plans[consumption_and_plans['plan_name']=='surf']
surf_plan = surf_plan.rename(columns={'duration':'surf_duration'})#cambiar nombre de columna
surf_plan = surf_plan.groupby('month')['surf_duration'].mean().reset_index()
#-------------------------------
ultimate_plan = consumption_and_plans[consumption_and_plans['plan_name']=='ultimate']
ultimate_plan = ultimate_plan.rename(columns={'duration':'ultimate_duration'})#cambiar nombre de columna
ultimate_plan = ultimate_plan.groupby('month')['ultimate_duration'].mean().reset_index()
#-------------------------------
#3. Hacemos un merge
plans_avg_call_duration = surf_plan.merge(ultimate_plan, on='month')
#print('plans_avg_call_duration:\n', plans_avg_call_duration)
#---------------------------
#4. Graficamos el dataframe resultante
plans_avg_call_duration.plot(x='month', kind='bar', rot=0)
plt.title('Duración promedio de llamadas por mes')
plt.xlabel('Mes')
plt.ylabel('Duración promedio de llamadas (min)')
plt.legend(['Surf', 'Ultimate'])
plt.ylim([100, 650])

plt.show()

In [ ]:
# Comparamos el número de minutos mensuales que necesitan los usuarios de cada plan. Posteriormente, trazamos un histograma.

# Comparamos minutos usados por mes, de cada plan
# Llamamos dos veces a plt.hist() para crear dos histogramas en el mismo gráfico
plt.hist(consumption_and_plans[consumption_and_plans['plan_name']=='surf']['duration'], bins=30, alpha=0.6, label='Surf')
plt.hist(consumption_and_plans[consumption_and_plans['plan_name']=='ultimate']['duration'], bins=30, alpha=0.6, label='Ultimate')
plt.title('Distribución de minutos usados por los usuarios')
plt.xlabel('Minutos usados por usuario')
plt.ylabel('Cantidad de usuarios')
plt.legend()
plt.show()



In [ ]:
# Calculamos la media y la varianza de la duración mensual de llamadas.
# Filtramos por plan y calculamos la media y la varianza respectivamente
surf_mean_duration = consumption_and_plans[consumption_and_plans['plan_name']=='surf']['duration'].mean()
surf_var_duration = np.var(consumption_and_plans[consumption_and_plans['plan_name']=='surf']['duration'], ddof=1)
print('Media de la duración mensual de llamadas del plan Surf:', surf_mean_duration)
print('Varianza de la duración mensual de llamadas del plan Surf:', surf_var_duration)
print()
ultimate_mean_duration = consumption_and_plans[consumption_and_plans['plan_name']=='ultimate']['duration'].mean()
ultimate_var_duration = np.var(consumption_and_plans[consumption_and_plans['plan_name']=='ultimate']['duration'], ddof=1)
print('Media de la duración mensual de llamadas del plan Ultimate:', ultimate_mean_duration)
print('Varianza de la duración mensual de llamadas del plan Ultimate:', ultimate_var_duration)

In [ ]:
# Trazamos un diagrama de caja para visualizar la distribución de la duración mensual de llamadas.

plt.boxplot([consumption_and_plans[consumption_and_plans['plan_name']=='surf']['duration'],
            consumption_and_plans[consumption_and_plans['plan_name']=='ultimate']['duration']], labels=['Surf', 'Ultimate'])
plt.title('Distribución de la duración mensual de llamadas')
plt.ylabel('Duración mensual de llamadas')
plt.ylim([-100, 1600])
plt.show()

Antes de continuar, es importante señalar que la cantidad de usuarios en el plan Surf es considerablemente mayor que en el plan Ultimate, lo cual puede influir en la comparación de comportamientos entre ambos grupos.

En cuanto al comportamiento de llamadas, se observa que la duración promedio en el plan Surf presenta un crecimiento gradual a lo largo del tiempo, mientras que en el plan Ultimate este aumento es menos uniforme. Durante los primeros meses existe una diferencia más marcada entre ambos planes; sin embargo, conforme avanza el tiempo, las tendencias tienden a converger.

Respecto a la distribución de minutos, en ambos planes la mayoría de los usuarios se mantiene por debajo de los 600 minutos mensuales. No obstante, el plan Ultimate concentra una mayor proporción de usuarios con consumos superiores a 1200 minutos.

Finalmente, el promedio de minutos utilizados es similar entre ambos planes, con valores de 445.77 para Surf y 443.27 para Ultimate.

### Mensajes

In [ ]:
# Comparamos el número de mensajes que tienden a enviar cada mes los usuarios de cada plan
# Comparamos la columna 'sms_id' por cada plan

monthly_messages_surf = consumption_and_plans[consumption_and_plans['plan_name']=='surf'].groupby('month')['sms_id'].sum()
monthly_messages_ultimate = consumption_and_plans[consumption_and_plans['plan_name']=='ultimate'].groupby('month')['sms_id'].sum()

monthly_sms_both_plans = pd.DataFrame({'Ultimate':monthly_messages_ultimate, 'Surf':monthly_messages_surf})

# print(monthly_sms_both_plans)

monthly_sms_both_plans.plot(kind='bar', rot=0)
plt.xlabel('Mes')
plt.ylabel('Cantidad de mensajes enviados por los usuarios')
plt.title('Mensajes enviados por mes')
plt.show()

In [ ]:
# Distribución de la columna 'sms_id'
# Llamamos dos veces a plt.hist() para crear dos histogramas en el mismo gráfico
plt.hist(consumption_and_plans[consumption_and_plans['plan_name']=='surf']['sms_id'], bins=30, alpha=0.6, label='Surf')
plt.hist(consumption_and_plans[consumption_and_plans['plan_name']=='ultimate']['sms_id'], bins=30, alpha=0.6, label='Ultimate')
plt.title('Distribución de mensajes enviados por los usuarios')
plt.xlabel('Mensajes enviados por usuario')
plt.ylabel('Cantidad de usuarios')
plt.legend()
plt.show()

In [ ]:
# Calculamos la media y la varianza de los mensajes enviados por los usuarios.
# Filtramos por plan y calculamos la media y la varianza respectivamente
surf_mean_messages = consumption_and_plans[consumption_and_plans['plan_name']=='surf']['sms_id'].mean()
surf_var_messages = np.var(consumption_and_plans[consumption_and_plans['plan_name']=='surf']['sms_id'], ddof=1)
print('Media de los mensajes enviados al mes por los usuarios del plan Surf:', surf_mean_messages)
print('Varianza de los mensajes enviados al mes por los usuarios del plan Surf:', surf_var_messages)
print()
ultimate_mean_messages = consumption_and_plans[consumption_and_plans['plan_name']=='ultimate']['sms_id'].mean()
ultimate_var_messages = np.var(consumption_and_plans[consumption_and_plans['plan_name']=='ultimate']['sms_id'], ddof=1)
print('Media de los mensajes enviados al mes por los usuarios del plan Ultimate:', ultimate_mean_messages)
print('Varianza de los mensajes enviados al mes por los usuarios del plan Ultimate:', ultimate_var_messages)

In [ ]:
# Trazamos un diagrama de caja para visualizar la distribución de los mensajes enviados por los usuarios

plt.boxplot([consumption_and_plans[consumption_and_plans['plan_name']=='surf']['sms_id'],
           consumption_and_plans[consumption_and_plans['plan_name']=='ultimate']['sms_id']],
            labels=['Surf', 'Ultimate'])
plt.title('Distribución de los mensajes enviados por los usuarios')
plt.ylabel('Mensajes enviados por los usuarios')
plt.ylim([-10, 300])
plt.show()

En relación con el envío de mensajes, se observa un incremento gradual en ambos planes a lo largo del tiempo. No obstante, el volumen de mensajes en el plan Surf tiende a ser consistentemente mayor en comparación con el plan Ultimate, ampliándose esta diferencia conforme avanzan los meses.

En cuanto a la distribución, la mayoría de los usuarios en ambos planes envía menos de 75 mensajes mensuales. Sin embargo, los niveles más altos de actividad (por encima de 200 mensajes) se concentran principalmente en usuarios del plan Surf.

Finalmente, el promedio de mensajes enviados es de 39.48 para el plan Surf y 46.53 para el plan Ultimate.

### Internet

In [ ]:
# Calculamos el consumo mensual de datos (GB) por plan para analizar su comportamiento a lo largo del tiempo

monthly_gb_surf = consumption_and_plans[consumption_and_plans['plan_name']=='surf'].groupby('month')['gb_used'].sum()
monthly_gb_ultimate = consumption_and_plans[consumption_and_plans['plan_name']=='ultimate'].groupby('month')['gb_used'].sum()

monthly_gb_both_plans = pd.DataFrame({'Surf':monthly_gb_surf, 'Ultimate':monthly_gb_ultimate})

monthly_gb_both_plans.plot(kind='bar', rot=0)
plt.xlabel('Mes')
plt.ylabel('Cantidad de GB usados por los usuarios')
plt.title('GB usados por mes')
plt.ylim([0, 5000])
plt.show()

In [ ]:
# Comparamos la cantidad de tráfico de Internet consumido por usuarios por plan
# Comparamos la columna 'gb_used' por plan
# Llamamos dos veces a plt.hist() para crear dos histogramas en el mismo gráfico
plt.hist(consumption_and_plans[consumption_and_plans['plan_name']=='surf']['gb_used'], bins=30, alpha=0.6, label='Surf')
plt.hist(consumption_and_plans[consumption_and_plans['plan_name']=='ultimate']['gb_used'], bins=30, alpha=0.6, label='Ultimate')
plt.title('Distribución de GB usados por los usuarios')
plt.xlabel('GB usados por usuario')
plt.ylabel('Cantidad de usuarios')
plt.legend()
plt.ylim([0, 250])
plt.show()

In [ ]:
# Calculamos la media y la varianza de los GB usados por los usuarios
# Filtramos por plan y calculamos la media y la varianza respectivamente
surf_mean_gb = consumption_and_plans[consumption_and_plans['plan_name']=='surf']['gb_used'].mean()
surf_var_gb = np.var(consumption_and_plans[consumption_and_plans['plan_name']=='surf']['gb_used'], ddof=1)
print('Media de los GB usados al mes por los usuarios del plan Surf:', surf_mean_gb)
print('Varianza de los GB usados al mes por los usuarios del plan Surf:', surf_var_gb)
print()
ultimate_mean_gb = consumption_and_plans[consumption_and_plans['plan_name']=='ultimate']['gb_used'].mean()
ultimate_var_gb = np.var(consumption_and_plans[consumption_and_plans['plan_name']=='ultimate']['gb_used'], ddof=1)
print('Media de los GB usados al mes por los usuarios del plan Ultimate:', ultimate_mean_gb)
print('Varianza de los GB usados al mes por los usuarios del plan Ultimate:', ultimate_var_gb)

In [ ]:
# Trazamos un diagrama de caja para visualizar la distribución de los GB usados por los usuarios

plt.boxplot([consumption_and_plans[consumption_and_plans['plan_name']=='surf']['gb_used'],
           consumption_and_plans[consumption_and_plans['plan_name']=='ultimate']['gb_used']],
           labels=['Surf', 'Ultimate'])
plt.title('Distribución de los GB usados por los usuarios')
plt.ylabel('GB usados por los usuarios')
plt.ylim([-5, 80])
plt.show()

En cuanto al consumo de datos, se observa un incremento progresivo en ambos planes a lo largo del tiempo. No obstante, el plan Surf presenta un mayor volumen de GB utilizados en comparación con el plan Ultimate, ampliándose esta diferencia conforme avanzan los meses.

Respecto a la distribución, la mayoría de los usuarios en ambos planes consume menos de 30 GB mensuales. Sin embargo, los niveles de consumo más altos (superiores a 65 GB) se concentran principalmente en usuarios del plan Surf.

Finalmente, el promedio de consumo es similar entre ambos planes, con valores de 16.75 GB para Surf y 17.37 GB para Ultimate.

## Ingreso

In [ ]:
# Calculamos el ingreso mensual de cada plan. Comparamos en un gráfico de barras.
# Filtramos por plan, agrupamos por mes y sacamos la suma de la columna 'monthly_income'
surf_monthly_income = consumption_and_plans[consumption_and_plans['plan_name']=='surf'].groupby('month')['monthly_income'].sum()
ultimate_monthly_income = consumption_and_plans[consumption_and_plans['plan_name']=='ultimate'].groupby('month')['monthly_income'].sum()
both_plans_monthly_income = pd.DataFrame({'Surf':surf_monthly_income, 'Ultimate':ultimate_monthly_income})
print(both_plans_monthly_income)
both_plans_monthly_income.plot(kind='bar', rot=0)
plt.title('Ingresos mensuales')
plt.xlabel('Mes')
plt.ylabel('Ingresos (dólares)')
plt.legend()
plt.ylim([0, 18000])
plt.show()

In [ ]:
# Distribución de los ingresos mensuales
# Llamamos dos veces a plt.hist() para crear dos histogramas en el mismo gráfico
plt.hist(consumption_and_plans[consumption_and_plans['plan_name']=='surf']['monthly_income'], bins=30, alpha=0.6, label='Surf')
plt.hist(consumption_and_plans[consumption_and_plans['plan_name']=='ultimate']['monthly_income'], bins=30, alpha=0.6, label='Ultimate')
plt.title('Distribución de ingresos de los usuarios')
plt.xlabel('Ingresos (dólares)')
plt.ylabel('Cantidad de usuarios')
plt.legend()
plt.ylim([0, 600])
plt.show()

In [ ]:
#Calculamos la media y varianza de los ingresos mensuales generados
    # Filtramos por plan y calculamos la media y la varianza respectivamente
surf_mean_income = consumption_and_plans[consumption_and_plans['plan_name']=='surf']['monthly_income'].mean()
surf_var_income = np.var(consumption_and_plans[consumption_and_plans['plan_name']=='surf']['monthly_income'], ddof=1)
print('Media de los ingresos mensuales generados por los usuarios del plan Surf:', surf_mean_income)
print('Varianza de los ingresos mensuales generados por los usuarios del plan Surf:', surf_var_income)
print()
ultimate_mean_income = consumption_and_plans[consumption_and_plans['plan_name']=='ultimate']['monthly_income'].mean()
ultimate_var_income = np.var(consumption_and_plans[consumption_and_plans['plan_name']=='ultimate']['monthly_income'], ddof=1)
print('Media de los ingresos mensuales generados por los usuarios del plan Ultimate:', ultimate_mean_income)
print('Varianza de los ingresos mensuales generados por los usuarios del plan Ultimate:', ultimate_var_income)

In [ ]:
# Vizualizamos la distribución de los ingresos con un diagrama de caja
plt.boxplot([consumption_and_plans[consumption_and_plans['plan_name']=='surf']['monthly_income'],
        consumption_and_plans[consumption_and_plans['plan_name']=='ultimate']['monthly_income']],
        labels=['Surf', 'Ultimate'])
plt.title('Distribución de los ingresos de los usuarios')
plt.ylabel('Ingresos de los usuarios')
plt.ylim([0, 700])
plt.show()

En términos de ingresos, ambos planes muestran un crecimiento gradual a lo largo del tiempo. Sin embargo, el plan Surf tiende a generar mayores ingresos mensuales en comparación con el plan Ultimate, ampliándose esta diferencia conforme avanzan los meses. Cabe destacar que únicamente en los primeros meses del año el plan Ultimate supera al plan Surf.

En cuanto a la distribución, la mayoría de los usuarios en ambos planes genera menos de 100 dólares mensuales. No obstante, los niveles más altos de ingresos (superiores a 200 dólares) se concentran principalmente en usuarios del plan Surf, incluyendo algunos casos atípicos con valores considerablemente elevados.

Finalmente, el ingreso promedio mensual es de 60.41 para el plan Surf y 72.25 para el plan Ultimate.

## Prueba de hipótesis estadísticas

In [ ]:
# Probamos las hipótesis:
# Hipótesis nula: Los ingresos promedio de los planes Surf y Ultimate son iguales
# Hipótesis alternativa: Los ingresos promedio de los planes Surf y Ultimate son diferentes (mayor o menor)
# Haremos una prueba de hipótesis bilateral ya que simplemente queremos saber si son diferentes (sin importar el sentido).
surf_income = consumption_and_plans[consumption_and_plans['plan_name']=='surf']['monthly_income'] # Filtramos por plan
ultimate_income = consumption_and_plans[consumption_and_plans['plan_name']=='ultimate']['monthly_income']# Filtramos por plan

# Establecemos la significación estadística
alpha = 0.05

# Determinamos si las varianzas son iguales con la prueba Levene
levene_test = st.levene(surf_income, ultimate_income)
equal_var = levene_test.pvalue > alpha

# Realizamos la prueba de hipótesis
results = st.ttest_ind(surf_income, ultimate_income, equal_var=equal_var) # Utilizamos ttest_ind ya que se trata de dos muestras independientes
print('Valor p:', results.pvalue)

if (results.pvalue < alpha):
    print('Rechazamos la hipótesis nula. Los ingresos promedio de los planes Surf y Ultimate son diferentes')
else:
    print('No podemos rechazar la hipótesis nula. No hay evidencia suficiente para determinar si los ingresos promedio de los planes Surf y Ultimate son diferentes')

In [ ]:
# Probamos las hipótesis
# Hipótesis nula: El ingreso promedio de los usuarios del área NY-NJ es igual al de los usuarios de otras regiones.
# Hipótesis alternativa: El ingreso promedio de los usuarios del área NY-NJ es diferente al de los usuarios de otras regiones.
# Realizaremos una prueba de hipótesis bilateral ya que solo queremos saber si las medias son diferentes (sin importar el sentido).

# Filtramos el dataframe por ciudad y seleccionamos la columna de ingresos mensuales
ny_nj_users_income = consumption_and_plans[consumption_and_plans['city'].str.contains('ny-nj', na=False)]['monthly_income'] # Llamamos a na=False en caso de que hayan valores ausentes (aunque no debería)
other_users_income = consumption_and_plans[~consumption_and_plans['city'].str.contains('ny-nj', na=False)]['monthly_income']

# Establecemos la significación estadística
alpha=0.05

# Determinamos si las varianzas son iguales con la prueba Levene
levene_test = st.levene(ny_nj_users_income, other_users_income)
equal_var = levene_test.pvalue > alpha

# Utilizamos ttest_ind debido a que los usuarios del área NY-NJ son totalmente independientes a los de otras áreas
results = st.ttest_ind(ny_nj_users_income, other_users_income, equal_var=equal_var)
print('Valor p:', results.pvalue)

if (results.pvalue < alpha):
    print('Rechazamos la hipótesis nula. El ingreso promedio de los usuarios del área NY-NJ es diferente al de los usuarios de otras regiones.')
else:
    print('No podemos rechazar la hipótesis nula. No hay evidencia suficiente para determinar si el ingreso promedio de los usuarios del área NY-NJ es diferente al de los usuarios de otras regiones.')

## Conclusión general

El análisis se desarrolló a partir de un proceso integral de preparación de datos, que incluyó la corrección de tipos de variables, estandarización de columnas, depuración de errores, así como la revisión de valores duplicados y ausentes. Posteriormente, se enriquecieron los datasets mediante la creación de nuevas variables relevantes que facilitaron el análisis.

Adicionalmente, se integraron múltiples fuentes de información en un único dataframe consolidado, lo que permitió analizar de manera conjunta el comportamiento de los usuarios en términos de consumo y generación de ingresos.

Para cada plan, se construyeron visualizaciones que permitieron analizar el consumo mensual de llamadas, mensajes y datos, utilizando gráficos de barras, histogramas y diagramas de caja. Asimismo, se calcularon métricas estadísticas como la media y la varianza para entender mejor la distribución y dispersión de estas variables.

De igual forma, se analizaron los ingresos mensuales generados por cada plan, comparando su evolución en el tiempo y su distribución entre usuarios, lo que permitió identificar diferencias clave en la generación de valor entre ambos planes.

Finalmente, se realizaron pruebas de hipótesis estadísticas, definiendo hipótesis nulas y alternativas, seleccionando el tipo de prueba adecuado en cada caso y estableciendo un nivel de significancia para validar los resultados obtenidos.

---

## Resultados de las gráficas

Es importante considerar que el número de usuarios del plan Surf es mayor que el del plan Ultimate, lo cual influye directamente en las diferencias observadas en el comportamiento y en los ingresos generados.

### Diferencia en minutos

Se observa que la duración promedio de llamadas en el plan Surf presenta un crecimiento más gradual a lo largo del tiempo en comparación con el plan Ultimate. Durante los primeros meses, existe una diferencia notable entre ambos planes; sin embargo, esta diferencia tiende a reducirse conforme avanza el año.

En cuanto a la distribución, la mayoría de los usuarios en ambos planes consume menos de 600 minutos mensuales. No obstante, el plan Ultimate presenta una mayor proporción de usuarios con consumos elevados (más de 1200 minutos), a pesar de contar con una menor base de usuarios.

---

### Diferencia en mensajes

En ambos planes, la cantidad de mensajes enviados muestra una tendencia creciente mes a mes. Sin embargo, el plan Surf mantiene consistentemente un mayor volumen de mensajes enviados en comparación con el plan Ultimate.

La distribución indica que la mayoría de los usuarios envía menos de 75 mensajes al mes. Además, los usuarios con un alto nivel de actividad (más de 200 mensajes) se concentran principalmente en el plan Surf.

---

### Diferencia en consumo de datos (GB)

El consumo de datos aumenta progresivamente en ambos planes a lo largo del tiempo. No obstante, el plan Surf presenta niveles de consumo superiores en comparación con el plan Ultimate.

En términos de distribución, la mayoría de los usuarios consume menos de 30 GB mensuales. Sin embargo, los consumos más altos (por encima de 65 GB) se concentran principalmente en usuarios del plan Surf.

---

### Diferencia en ingresos

Aunque ambos planes muestran una tendencia creciente en ingresos mensuales, el plan Surf genera mayores ingresos totales en la mayoría de los meses. Solo en los primeros meses del año el plan Ultimate supera al plan Surf, pero posteriormente este último predomina de forma consistente.

En cuanto a la distribución, la mayoría de los usuarios genera menos de 100 dólares al mes. Sin embargo, los ingresos más altos (más de 200 dólares mensuales) se concentran en el plan Surf. Incluso, se identificaron casos específicos de usuarios con ingresos significativamente superiores dentro de este plan.

---

## Resultados de hipótesis

### Primera hipótesis

Se evaluó si existían diferencias en el ingreso promedio entre los planes Surf y Ultimate, utilizando dos muestras independientes y una prueba bilateral.

Los resultados indican que existe una diferencia estadísticamente significativa entre ambos planes, por lo que se rechaza la hipótesis nula.

---

### Segunda hipótesis

Se analizó si el ingreso promedio de los usuarios del área NY-NJ difiere del de otras regiones, utilizando nuevamente muestras independientes y una prueba bilateral.

Los resultados muestran que también existe una diferencia estadísticamente significativa, lo que permite rechazar la hipótesis nula.

---

## Conclusión final

Los resultados obtenidos evidencian diferencias claras en el comportamiento de los usuarios y en la generación de ingresos entre los planes analizados. El plan Surf destaca por concentrar una mayor actividad de uso y generar mayores ingresos totales, impulsado en gran medida por su mayor base de usuarios y la presencia de consumidores intensivos.

Por su parte, el plan Ultimate presenta un ingreso promedio competitivo, pero una menor participación en términos de volumen, lo que limita su contribución total.

Las pruebas estadísticas respaldan estos hallazgos, confirmando que las diferencias observadas no son producto del azar, sino que reflejan patrones reales en los datos.

Con base en lo anterior, se recomienda orientar las estrategias comerciales hacia la optimización del plan Surf, aprovechando su alto nivel de adopción y consumo. Asimismo, el plan Ultimate podría beneficiarse de ajustes en su posicionamiento o en su propuesta de valor para incrementar su adopción y mejorar su impacto en los ingresos totales.